# Requirements

In [11]:
# Libraries
import rasterio
import geopandas as gpd

from beak.utilities.raster_processing import calculate_distance_from_raster, calculate_distance_from_raster_gdal_base_fn
from beak.utilities.io import data_folder, save_raster
from beak.utilities.conversions import create_binary_raster

# Set base data folder path
data_folder = data_folder()

# Paths
raster_file = data_folder / "BASE_RASTERS" / "EPSG_102008_RES_2240_STATE_ALASKA.tif"
base_raster_file = data_folder / "BASE_RASTERS" / "EPSG_102008_RES_2240_STATE_ALASKA.tif"
vector_file = data_folder / "RAW" / "templates" / "features_distance_calculations.gpkg"


In [32]:
from rasterio.crs import CRS

raster = rasterio.open(raster_file)
crs = raster.crs

if crs.is_epsg_code:
    epsg = CRS.to_epsg(crs)
    crs = CRS.from_epsg(epsg)
    
print(epsg, crs)


None ESRI:102008


In [33]:
crs = CRS.from_epsg(4326)
print(crs)

EPSG:4326


# Distance calculations

Calculate the distance/proximity of cells in a raster based on the the GDAL proximity function. <br>
This **example** shows ways how to accomplish this using different input data types for both **raster** and **vector** data. <p>
Examples shown are for **binary** rasters.<p>
The function `calculate_distance_from_raster()` works for the three input types 
- file
- raster object
- array and metadata

Additionally, by providing a mask (such as a base raster is), the function allows to replace the values in the distance raster with the respective **nodata** values, resulting in the same shape as the mask raster. <br> The mask must be a `rasterio.io.DatasetReader` object.

## 1. From raster

### 1.1 File

In [ ]:
# Use a raster file as input and return an array and metadata
distance_array, distance_meta = calculate_distance_from_raster(input_path=raster_file)
save_raster("distance_from_raster_file.tif", distance_array, metadata=distance_meta, overwrite=True)

If you have a raster file ready and directly want to save the output, you can also use the original GDAL way. <br>
However, you need to provide **additional parameters** to get reliable results and it is **not** possible to apply a conversion factor to the result, unless the saved raster will be opened and saved/overwritten again.


In [ ]:
# Use a raster file and directly write an output distance raster
calculate_distance_from_raster_gdal_base_fn(input_path=raster_file, output_path="distance_from-raster_file_gdal_base_fn.tif", values=1, distunits="GEO")

### 1.2 Dataset

In [ ]:
# Open file and create a raster object
raster_dataset = rasterio.open(raster_file)

# Use the raster object as input and save
distance_array, distance_meta = calculate_distance_from_raster(input_raster=raster_dataset)
save_raster("distance_from_raster_dataset.tif", distance_array, metadata=distance_meta, overwrite=True)

### 1.3 Array and metadata

In [ ]:
# Open file and create raster object
raster_dataset = rasterio.open(raster_file)

# Read raster data and metadata from the raster object
raster_array = raster_dataset.read()
raster_meta = raster_dataset.meta.copy()

# Use the raster data and metadata as input and save
distance_array, distance_meta = calculate_distance_from_raster(input_data=(raster_array, raster_meta))
save_raster("distance_from_raster_array.tif", distance_array, metadata=distance_meta, overwrite=True)

## 2. From vector 

In [ ]:
# Load data into geodataframe
gdf = gpd.read_file(vector_file, layer="Nat_GY_Mag_DeepMagSources_Worms_USCanada")

# Convert the geodataframe's crs to the base raster's crs
raster = rasterio.open(base_raster_file)
gdf = gdf.to_crs(raster.crs)

# Convert geodataframe to binary raster incl. metadata plus applying a base raster for maintaining the extent
binary_raster, meta = create_binary_raster(geodataframe=gdf,
                                           base_raster=raster,
                                           query=None,
                                           return_meta=True,
                                           )

# Calculate distance from raster
distance_array, distance_meta = calculate_distance_from_raster(input_data=(binary_raster, meta), input_mask=raster)
save_raster("distance_from_vector_conversion.tif", distance_array, metadata=distance_meta, overwrite=True)
